# Kuna-Chen, Chen ML  Dimensionality Reduction_test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import seaborn as sns


from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA


from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV,  KFold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.datasets import cifar10
from joblib import dump
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)

In [ ]:
# Load data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# CIFAR-10 has 10 classes
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def visualize_cifar10_samples(X, y, class_names, samples_per_class,random_seed = 2025):
    """
    Displays a grid of sample images for each class in CIFAR-10.
    X: image array, shape (N, 32, 32, 3)
    y: labels array, shape (N, 1)
    class_names: list of 10 class names
    samples_per_class: how many examples to show per class
    """
    np.random.seed(random_seed)
    y = y.squeeze()  # Convert from shape (N,1) to (N,) if needed
    num_classes = len(class_names)
    plt.figure(figsize=(samples_per_class * 2, num_classes * 2))
    for cls_idx in range(num_classes):
        idxs = np.flatnonzero(y == cls_idx)
        idxs = np.random.choice(idxs, samples_per_class, replace=False)
        for i, idx in enumerate(idxs):
            plt.subplot(num_classes, samples_per_class, cls_idx * samples_per_class + i + 1)
            plt.imshow(X[idx])
            plt.axis('off')
            if i == 0:
                plt.title(class_names[cls_idx])
    plt.tight_layout()
    plt.show()

# Example usage:
visualize_cifar10_samples(X_train, y_train, class_names,5)


In [ ]:
Model0 = load('model_svc.joblib')
Model1 = load('model_RanF.joblib')
Model2 = load('model_LogR.joblib')
Model3 = load('model_pca_svc.joblib')
Model4 = load('model_pca_RanF.joblib')
Model5 = load('model_pca_LogR.joblib')
Model6 = load('model_tsne_svc.joblib')
Model7 = load('model_umap_svc.joblib')

In [ ]:
models = [Model0, Model1, Model2, Model3, Model4, Model5, Model6, Model7]
model_names = ['SVC', 'Random Forest', 'Logistic Regression', 
               'PCA+SVC', 'PCA+Random Forest', 'PCA+Logistic Regression',
               't-SNE+SVC', 'UMAP+SVC']

In [ ]:
results = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'confusion_matrices': [],
    'fpr': [],
    'tpr': [],
    'auc': []
}
plt.figure(figsize=(10, 8))

In [ ]:
for i, (model, name) in enumerate(zip(models, model_names)):
    print(f"Evaluating {name}...")
    y_pred = model.predict(X_test)

    try:
        y_prob = model.predict_proba(X_test)[:, 1]  
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            print(f"  Warning: Could not get probability estimates for {name}")
            y_prob = np.ones_like(y_pred)  
    
    # 1. Accuracy - Proportion of correct predictions
    acc = accuracy_score(y_test, y_pred)
    results['accuracy'].append(acc)
    
    # 2. Precision 
    prec = precision_score(y_test, y_pred, average='weighted')
    results['precision'].append(prec)
    
    # 3. Recall
    rec = recall_score(y_test, y_pred, average='weighted')
    results['recall'].append(rec)
    
    # 4. F1 Score 
    f1 = f1_score(y_test, y_pred, average='weighted')
    results['f1'].append(f1)
    
    # 5. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    results['confusion_matrices'].append(cm)
    

    
    # Print all metrics for this model
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  Detailed classification report:")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (cm, name) in enumerate(zip(results['confusion_matrices'], model_names)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'Confusion Matrix - {name}')
    axes[i].set_xlabel('Predicted Labels')
    axes[i].set_ylabel('True Labels')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300)
plt.show()

In [ ]:


# Create a bar chart comparing all models across metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
metric_values = [
    results['accuracy'],
    results['precision'],
    results['recall'],
    results['f1']
]
x = np.arange(len(model_names))
width = 0.2
fig, ax = plt.subplots(figsize=(12, 6))


In [ ]:
for i, (metric, values) in enumerate(zip(metrics, metric_values)):
    ax.bar(x + i*width, values, width, label=metric)

ax.set_xlabel('Models')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()

In [ ]:
print("SUMMARY OF ALL MODEL METRICS:")
print("-" * 50)
print("Model      | Accuracy | Precision | Recall   | F1 Score | AUC")
print("-" * 80)
for i, name in enumerate(model_names):
    auc_value = results['auc'][i] if i < len(results['auc']) else float('nan')
    print(f"{name:11} | {results['accuracy'][i]:.6f} | {results['precision'][i]:.6f} | {results['recall'][i]:.6f} | {results['f1'][i]:.6f} | {auc_value:.6f}")